In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
%run ../00-common/02.bronze_helpers

In [0]:
catalog_name

In [0]:
source_file=f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name=f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,DoubleType

circuit_Schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType())
])

In [0]:
circuits_df=(
    spark.read
    .format('csv')
    .option('header','true')
    #.option('inferSchema','true')
    .option('mode','FAILFAST')
    .schema(circuit_Schema)
    .load(source_file)
)

In [0]:
circuits_df.show(5)

In [0]:
display(circuits_df)

In [0]:

circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

In [0]:
#circuits_final_df=circuits_final_df.withColumn("batch_id",F.lit(v_batch_id))

In [0]:
# (
#     circuits_final_df
#     .write
#     .format('delta')
#     .mode('overwrite')
#     .partitionBy('batch_id')
#     .option('replaceWhere', f"batch_id='{v_batch_id}'")
#     .saveAsTable(table_name)
# )

In [0]:
write_to_bronze(
    input_df=circuits_final_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.table(table_name))